# RAG (Retrieval Augmented Generation)
#### Importing Dependencies.
Make sure to create an env using <b> python 3.10 </b> to prevent facing any library issue.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
from tqdm import tqdm
import hazm

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

from torch.utils.data import DataLoader, Dataset 

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"This device supports '{device}'")

This device supports 'cuda'


_______________________________
#### Loading Embedding Model
* Link: https://huggingface.co/heydariAI/persian-embeddings

In [2]:
embedding_model_path = os.path.join("C:/Users/Amir/Desktop/Models/Embeddings/Persian Embedding")

# Loading model directly
from transformers import AutoTokenizer, AutoModel
embedding_tokenizer = AutoTokenizer.from_pretrained("heydariAI/persian-embeddings", cache_dir=embedding_model_path)
embedding_model = AutoModel.from_pretrained("heydariAI/persian-embeddings", cache_dir=embedding_model_path)
embedding_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def text_to_embedding(text, embedding_tokenizer, embedding_model):
    encoded_input = embedding_tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=512)
    with torch.inference_mode():
        model_output = embedding_model(**encoded_input)
    
    embedding = mean_pooling(model_output, encoded_input["attention_mask"])
    return(embedding.cpu().numpy())

In [3]:
input_text = input("User input: ")
embedded_input = text_to_embedding(input_text, embedding_tokenizer, embedding_model)

TOP_K = 3

from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)

results = client.query_points(
    collection_name="wiki_farsi",
    query=embedded_input.squeeze().tolist(),
    limit=TOP_K
)

contexts = [p.payload["text"] for p in results.points]

User input: برج میلاد در چه سالی ساخته شد؟


______________________________________________________________
#### Defining Initial Prompt for LLM using Retrieved Information

In [4]:
query = input_text
retrieved_data = contexts[0]
prompt = f"""
اطلاعات زیر از پایگاه داده استخراج شده است:
{retrieved_data}
با استفاده از اطلاعات بالا به سوال زیر پاسخ بده:
{query}
"""

_________
#### LLM 

In [5]:
# === Never understimate documentations === #

# Testing the output of the LLM (Qwen3) Loaded locally
API_TOKEN = "sk-lm-MgYjkxo3:ybum05qNOcyz9xmALaSd"
LLM_MODEL = "qwen3-14b"

# WARNING: Double check the host address to prevent facing any errors
HOST = "http://172.30.144.1:6666"

import os
import requests
import json

response = requests.post(
    f"{HOST}/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json"
    },
    json={
        "model": LLM_MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.1, # Creativity parameter
        "top_p":0.9, # Limiting the sampling space
        "max_tokens": 4096 # Maximum length of output
    }
)

data = response.json()

print("\nAssistant:")
print(data["choices"][0]["message"]["content"])


Assistant:


برج میلاد در **۱۳۸۷** (مطابق تقویم شمسی) با شعار «آسمان نزدیک است» گشایش یافت. این تاریخ به معنای **۲۰۰۸ میلادی** در تقویم گریгорی است، اما بر اساس اطلاعات ارائه‌شده که از تقویم شمسی استفاده می‌کند، سال ساخت و بهره‌برداری آن **۱۳۸۷** می‌باشد.
